In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi")

!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [3]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

Tensorflow is not available


In [4]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [5]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [6]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:
import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        

        # 1. Suggerisci la metrica
        # similarity = "tversky"

        topK = optuna_trial.suggest_int("topK", 5, 15)
        shrink = optuna_trial.suggest_int("shrink", 90, 120)

        # 3. Parametri condizionali per le metriche avanzate
        args = {} # Dizionario per gli argomenti extra

        args["tversky_alpha"] = optuna_trial.suggest_float("tversky_alpha", 0.0, 0.3)
        args["tversky_beta"] = optuna_trial.suggest_float("tversky_beta", 1.6, 1.9)

        """elif similarity == "asymmetric":
            # Asymmetric cosine ha un parametro alpha (peso del denominatore)
            # alpha=0.5 è simile alla cosine standard
            args["asymmetric_alpha"] = optuna_trial.suggest_float("asymmetric_alpha", 0.0, 2.0)"""

        # 4. Passa tutto al fit
        recommender_instance = ItemKNNCFRecommender(URM_combined)
        recommender_instance.fit(topK=topK, 
                                shrink=shrink, 
                                similarity="tversky", 
                                feature_weighting="TF-IDF", 
                                **args) # Scompatta gli argomenti extra
        

        """#Cambiare il modello qui sotto, insieme al range e ai parametri 
        recommender_instance = ItemKNNCFRecommender(URM_combined)
        recommender_instance.fit(
                            topK = optuna_trial.suggest_int("topK", 50, 80),
                            shrink = optuna_trial.suggest_int("shrink", 0, 1000),
                            similarity = optuna_trial.suggest_categorical("similarity", ["cosine", "jaccard", "dice", "tversky", "asymmetric"]),
                            feature_weighting = "TF-IDF"
                             )"""
        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)

In [9]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 50)

[I 2025-12-08 12:45:33,631] A new study created in memory with name: no-name-e5c64e7d-c77a-4de2-b74c-55599bdb6747


Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1550.53 column/sec. Elapsed time 4.49 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 4.29 sec. Users per second: 6311
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1789.97 column/sec. Elapsed time 3.89 sec
EvaluatorHoldout: Ignoring 24 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27071 (100.0%) in 4.12 sec. Users per second: 6565
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1985.51 column/sec. Elapsed time 3.51 sec
EvaluatorHoldout: Ignoring 27 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27068 (100.0%) in 4.05 sec. Users per second: 6678
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1974.

[I 2025-12-08 12:46:13,796] Trial 0 finished with value: 0.2484005967223998 and parameters: {'topK': 13, 'shrink': 90, 'tversky_alpha': 0.24745253840543824, 'tversky_beta': 1.8174944387611558}. Best is trial 0 with value: 0.2484005967223998.


[0.2495076969493823, 0.24945199553375588, 0.2475436679966274, 0.24894080302202456, 0.24655882011020883]
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1998.27 column/sec. Elapsed time 3.49 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 3.86 sec. Users per second: 7009
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 2021.63 column/sec. Elapsed time 3.45 sec
EvaluatorHoldout: Ignoring 24 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27071 (100.0%) in 3.90 sec. Users per second: 6945
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1991.35 column/sec. Elapsed time 3.50 sec
EvaluatorHoldout: Ignoring 27 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27068 (100.0%) in 3.88 sec. Users per second: 6

[I 2025-12-08 12:46:51,349] Trial 1 finished with value: 0.25010885394001303 and parameters: {'topK': 5, 'shrink': 101, 'tversky_alpha': 0.2317995341152158, 'tversky_beta': 1.699662272963801}. Best is trial 1 with value: 0.25010885394001303.


...

[0.2518759118098205, 0.25125607689046997, 0.25106189928777045, 0.2533042401983017, 0.2511675354340009]
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 2089.04 column/sec. Elapsed time 3.34 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 3.86 sec. Users per second: 7013
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 2093.00 column/sec. Elapsed time 3.33 sec
EvaluatorHoldout: Ignoring 24 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27071 (100.0%) in 3.87 sec. Users per second: 6987
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 2082.56 column/sec. Elapsed time 3.35 sec
EvaluatorHoldout: Ignoring 27 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27068 (100.0%) in 3.86 sec. Users per second: 70

[I 2025-12-08 13:16:00,692] Trial 48 finished with value: 0.2495941802367711 and parameters: {'topK': 8, 'shrink': 100, 'tversky_alpha': 0.08223145618092166, 'tversky_beta': 1.8064323685142856}. Best is trial 20 with value: 0.2520214853277825.


[0.25024555422818956, 0.24953763389093642, 0.24883511832668598, 0.25041693031647894, 0.24893566442156476]
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 2067.44 column/sec. Elapsed time 3.37 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 3.95 sec. Users per second: 6849
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 2086.87 column/sec. Elapsed time 3.34 sec
EvaluatorHoldout: Ignoring 24 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27071 (100.0%) in 3.96 sec. Users per second: 6831
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 6969 (100.0%), 1990.03 column/sec. Elapsed time 3.50 sec
EvaluatorHoldout: Ignoring 27 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27068 (100.0%) in 3.95 sec. Users per second:

[I 2025-12-08 13:16:38,034] Trial 49 finished with value: 0.25099317074720734 and parameters: {'topK': 9, 'shrink': 102, 'tversky_alpha': 0.20567657769338862, 'tversky_beta': 1.607218732842503}. Best is trial 20 with value: 0.2520214853277825.


[0.2511324058768966, 0.2510744554399338, 0.2511333188547608, 0.2513941386572707, 0.2502315349071747]


In [10]:
optuna_study.best_trial.params

{'topK': 8,
 'shrink': 110,
 'tversky_alpha': 0.1811461557442943,
 'tversky_beta': 1.780993258524839}

In [11]:
save_results.results_df

,result,train_time (min)
0,0.248401,0.669391
1,0.250109,0.625831
2,0.244835,0.637662
3,0.249293,0.647329
4,0.250337,0.627337
5,0.250401,0.641942
6,0.251168,0.631441
7,0.248177,0.653608
8,0.251125,0.627661
9,0.246279,0.621987


In [12]:
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances


plot_param_importances(optuna_study)

In [13]:
plot_parallel_coordinate(optuna_study, params=["topK", "shrink", "tversky_alpha", "tversky_beta"])

In [14]:
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
best_hyperparams



{}